# Cone

**Módulo 06 — Geometria Espacial** · Notebook 12 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0612a_cone.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0610a_piramide.ipynb`](0610a_piramide.ipynb) e [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb)
- Módulo 04 — Geometria Plana: [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (comprimento da circunferência e de um arco) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (área do círculo e do setor circular)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- reconhecer superfícies cônicas e os elementos de um cone;
- classificar cones e identificar suas seções meridiana e transversal;
- usar a planificação do cone reto para deduzir as áreas lateral e total;
- deduzir e usar a fórmula do volume do cone.

**Tempo estimado:** 70 a 90 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math
import numpy as np

## Como fazer um chapéu de festa?

Pegue um disco de papel, recorte uma "fatia" (um setor circular) e junte as duas bordas. Pronto: você tem um chapéu de festa, um cone!

Mas que fatia recortar? Se você deixar um setor grande (quase o disco inteiro), o chapéu fica largo e baixinho. Se deixar um setor pequeno, ele fica fino e alto.

🕹️ No widget abaixo, escolha o tamanho do papel (o raio do setor) e o ângulo do setor que você vai usar. Depois, use o controle "enrolar" para montar o chapéu.

In [2]:
def montar_chapeu(raio_do_papel: int, angulo_do_setor: int, enrolar: float) -> None:
    """Enrola o setor de papel escolhido até ele virar um chapéu de festa (um cone)."""
    comprimento_do_arco = 2 * math.pi * raio_do_papel * angulo_do_setor / 360
    # O arco vira a circunferência da base: 2π · raio_do_chapeu = comprimento_do_arco.
    raio_do_chapeu = comprimento_do_arco / (2 * math.pi)
    # O raio do papel vira a geratriz: Pitágoras dá a altura.
    altura_do_chapeu = math.sqrt(raio_do_papel**2 - raio_do_chapeu**2)

    fig = u3d.criar_figura(titulo="Do setor de papel ao chapéu")
    fig = u3d.desenhar_enrolamento_do_cone(fig, raio_do_papel, angulo_do_setor, enrolar)
    if enrolar == 0:
        # Visto de cima, o setor aparece em verdadeira grandeza.
        fig = u3d.definir_camera(fig, olho=(0, -0.4, 2.2))
    else:
        fig = u3d.definir_camera(fig, olho=(1.5, -1.5, 0.7))
    u3d.mostrar(fig)

    print(f"Comprimento do arco: {comprimento_do_arco:.2f} cm")
    print(f"Raio do chapéu: {raio_do_chapeu:.2f} cm")
    print(f"Altura do chapéu: {altura_do_chapeu:.2f} cm")
    print(f"Pedaço recortado do disco: {360 - angulo_do_setor}°")


interact(
    montar_chapeu,
    raio_do_papel=widgets.IntSlider(
        min=5, max=30, step=1, value=15, description="raio (cm)",
        continuous_update=False,
    ),
    angulo_do_setor=widgets.IntSlider(
        min=30, max=330, step=6, value=216, description="ângulo (°)",
        continuous_update=False,
    ),
    enrolar=widgets.FloatSlider(
        min=0, max=1, step=0.1, value=0, description="enrolar",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=15, continuous_update=False, description='raio (cm)', max=30, min=5), In…

Repare em três coisas:
- a **borda curva** do setor vira a **circunferência da base** do chapéu;
- o **raio do papel** vira a **geratriz** do cone, a distância da ponta até a borda;
- quanto **mais** papel você recorta, mais pontudo fica o chapéu. É o mesmo "defeito" do notebook 06: para fazer uma ponta, é preciso tirar um pedaço do papel.

Neste notebook você vai transformar essas observações em fórmulas para as áreas e o volume do cone.

## 1. Superfícies cônicas

Imagine uma reta que passa sempre por um ponto fixo $V$ e se move apoiada numa curva. O "rastro" dessa reta é uma **superfície cônica**.

> 📌 **Definição.**
> - O ponto fixo $V$ é o **vértice**.
> - A curva em que a reta se apoia é a **diretriz**.
> - Cada posição da reta é uma **geratriz**.

Como as geratrizes são **retas** (infinitas para os dois lados), a superfície cônica tem **duas folhas**, uma de cada lado do vértice, como uma ampulheta infinita. Quando a diretriz é uma circunferência e o vértice fica na reta perpendicular ao seu plano pelo centro, a superfície é **cônica circular reta**.

**No dia a dia:** funis, casquinhas de sorvete, abajures, cones de trânsito, o feixe de luz de uma lanterna.

### As seções cônicas

Corte uma superfície cônica de duas folhas por um plano. Dependendo da inclinação do plano, aparecem quatro curvas diferentes:

| Inclinação do plano | Curva |
|---|---|
| perpendicular ao eixo | **circunferência** |
| menos inclinado que as geratrizes | **elipse** (curva fechada, "achatada") |
| paralelo a uma geratriz | **parábola** (curva aberta) |
| mais inclinado que as geratrizes | **hipérbole** (dois ramos, um em cada folha) |


> 🕵️ **Curiosidade histórica.** O grego **Apolônio de Perga** (por volta de 200 a.C.) estudou essas curvas em profundidade numa obra chamada *As Cônicas*, e foi ele quem popularizou os nomes elipse, parábola e hipérbole. Quase dois mil anos depois, **Kepler** descobriu que os planetas giram em volta do Sol em órbitas **elípticas**. Hoje, as parábolas aparecem nas antenas parabólicas e nos faróis de carro.

> 💡 **Dica.** Acenda uma lanterna (ou um abajur com cúpula aberta) perto de uma parede. O feixe de luz é um cone; a parede é o plano de corte. Mudando o ângulo da lanterna, você vê na parede uma circunferência, uma elipse, uma parábola ou um ramo de hipérbole!

🕹️ Incline o plano de corte no widget abaixo.

In [3]:
def cortar_cone_duplo(
    semiabertura: int, inclinacao_do_plano: int, mostrar_cone: bool
) -> None:
    """Corta o cone duplo por um plano inclinado e classifica a curva obtida."""
    angulo_das_geratrizes = 90 - semiabertura  # ângulo entre as geratrizes e a horizontal
    if inclinacao_do_plano == 0:
        curva = "circunferência"
    elif inclinacao_do_plano < angulo_das_geratrizes:
        curva = "elipse"
    elif inclinacao_do_plano == angulo_das_geratrizes:
        curva = "parábola"
    else:
        curva = "hipérbole"

    # O plano de corte passa pelo ponto (0.5, 0, 1), fora do eixo, para nunca passar
    # pelo vértice. Seu vetor normal (perpendicular ao plano) é (sen i, 0, cos i): com
    # i = 0° ele aponta para cima e o plano é horizontal; com i = 90° ele aponta para o
    # lado e o plano é vertical. Seno e cosseno decompõem o vetor, como no ciclo
    # trigonométrico.
    i = math.radians(inclinacao_do_plano)
    plano = ((0.5, 0, 1), (math.sin(i), 0, math.cos(i)))
    trechos = u3d.secao_plana_do_cone_duplo(semiabertura, plano, altura=3)

    fig = u3d.criar_figura(titulo=f"Seção: {curva}")
    if mostrar_cone:
        # A diretriz: a circunferência na altura 3, onde o raio é 3 · tan(semiabertura).
        raio_no_topo = 3 * math.tan(math.radians(semiabertura))
        diretriz = u3d.poligono_regular(80, raio=raio_no_topo, centro=(0, 0, 3))
        fig = u3d.desenhar_superficie_conica(
            fig, (0, 0, 0), diretriz + diretriz[:1], opacidade=0.25, prolongar=1
        )

    # O pedaço do plano fica centrado na curva e é grande o bastante para contê-la.
    pontos = [p for trecho in trechos for p in trecho]
    centro = np.mean(pontos, axis=0)
    tamanho = 2 * max(u3d.medir_distancia(pontos, centro)) + 1
    fig = u3d.desenhar_plano(fig, centro, plano[1], opacidade=0.15, tamanho=tamanho)
    for trecho in trechos:
        fig = u3d.desenhar_curva_no_espaco(
            fig, trecho, cor=u3d.CORES["destaque"], espessura=6
        )
    fig = u3d.desenhar_ponto(fig, (0, 0, 0), rotulo="V")
    u3d.mostrar(fig)

    print(f"Ângulo entre as geratrizes e a horizontal: {angulo_das_geratrizes}°")
    print(f"Inclinação do plano: {inclinacao_do_plano}°")
    print(f"Curva: {curva}")
    print(f"Trechos encontrados no pedaço desenhado: {len(trechos)}")
    if curva == "elipse" and trechos[0][0] != trechos[0][-1]:
        print("(A elipse é fechada, mas parte dela fica fora do pedaço de cone desenhado.)")
    if curva == "hipérbole" and len(trechos) < 2:
        print("(O outro ramo existe, mas fica fora do pedaço de cone desenhado.)")


interact(
    cortar_cone_duplo,
    semiabertura=widgets.IntSlider(
        min=15, max=45, step=5, value=30, description="semiabertura (°)",
        continuous_update=False,
    ),
    inclinacao_do_plano=widgets.IntSlider(
        min=0, max=90, step=5, value=20, description="inclinação (°)",
        continuous_update=False,
    ),
    mostrar_cone=widgets.Checkbox(value=True, description="Mostrar o cone"),
);

interactive(children=(IntSlider(value=30, continuous_update=False, description='semiabertura (°)', max=45, min…

**Pare e pense**

1. Num cone duplo com semiabertura de 30°, as geratrizes formam quantos graus com a horizontal? Que curva aparece com um plano inclinado 60°?
2. Por que a hipérbole tem dois ramos e a elipse só um?

<details>
<summary>Ver respostas</summary>

1. $90° - 30° = 60°$. Com o plano inclinado 60°, ele fica paralelo a uma geratriz: aparece uma parábola.
2. O plano da hipérbole é tão inclinado que corta **as duas folhas** do cone. O plano da elipse corta só uma folha, e de um lado a outro dela.
</details>

> 📎 **Conexão.** As cônicas serão estudadas com equações no módulo 07 (Geometria Analítica). Daqui para a frente, neste notebook, vamos trabalhar só com **uma folha**, limitada por uma base: o cone.

## 2. Cone circular

> 📌 **Definição.** Considere um círculo de centro $O$ e raio $r$ num plano $\alpha$ e um ponto $V$ fora de $\alpha$. O **cone circular** é o sólido formado por todos os segmentos que ligam $V$ aos pontos do círculo.

**Elementos**
- **Vértice** $V$ e **base** (o círculo de raio $r$).
- **Eixo:** o segmento $VO$.
- **Geratrizes:** os segmentos que ligam $V$ aos pontos da circunferência da base.
- **Altura** $h$: a distância de $V$ ao plano da base.

### Classificação

- **Cone reto:** o eixo é perpendicular à base. Então a altura é o próprio eixo, todas as geratrizes têm o mesmo comprimento $g$ e, pelo triângulo retângulo $VOA$ (com $A$ na circunferência):
  $$g^2 = h^2 + r^2$$
  O cone reto também é chamado **cone de revolução**: é gerado por um **triângulo retângulo girando** em torno de um dos catetos.
- **Cone oblíquo:** o eixo é oblíquo à base. As geratrizes têm comprimentos diferentes.

> ⚠️ **Erro comum.** Confundir a altura $h$ com a geratriz $g$. No cone reto, a geratriz é a hipotenusa do triângulo $VOA$: ela é sempre **maior** que a altura.

### Seções

- **Seção transversal** (plano paralelo à base): um **círculo**. A uma distância $d$ do vértice, o raio da seção é $r \cdot \frac{d}{h}$, como as seções paralelas da pirâmide (notebook 10).
- **Seção meridiana** (plano que contém o eixo): no cone reto, um **triângulo isósceles** de base $2r$, altura $h$ e lados iguais a $g$.

> 📌 **Definição.** Um **cone equilátero** é um cone reto cuja seção meridiana é um **triângulo equilátero**: $g = 2r$ e, portanto, $h = r\sqrt{3}$.

🕹️ Monte um cone e escolha a seção.

In [4]:
def explorar_cone(
    raio: float, altura: float, desvio_do_vertice: float, giro_da_secao: int, secao: str
) -> None:
    """Desenha um cone, classifica-o e mostra a seção escolhida."""
    vertice = (desvio_do_vertice, 0, altura)
    fig = u3d.criar_figura(titulo="O cone e suas seções")
    fig = u3d.desenhar_cone(
        fig, vertice, (0, 0, 0), raio, opacidade=0.3, mostrar_eixo=True,
        mostrar_geratrizes=8,
    )

    avisos = []
    if secao == "Transversal":
        # Na metade da altura, estamos a d = h/2 do vértice: o raio é r · d / h = r / 2.
        distancia_ao_vertice = altura / 2
        raio_da_secao = raio * distancia_ao_vertice / altura
        # O centro da seção fica sobre o eixo VO, no meio do caminho.
        centro_da_secao = (desvio_do_vertice / 2, 0, altura / 2)
        circulo = u3d.poligono_regular(60, raio=raio_da_secao, centro=centro_da_secao)
        fig = u3d.desenhar_poligono(
            fig, circulo, cor=u3d.CORES["destaque"], opacidade=0.6
        )
        avisos.append(f"Seção transversal: círculo de raio {raio_da_secao:.2f}")
    elif secao == "Meridiana":
        triangulo = u3d.secao_meridiana_do_cone(
            vertice, (0, 0, 0), raio, giro=giro_da_secao
        )
        fig = u3d.desenhar_poligono(
            fig, triangulo, cor=u3d.CORES["destaque"], opacidade=0.6
        )
        A, B, V = triangulo
        lados = [u3d.medir_distancia(V, A), u3d.medir_distancia(V, B),
                 u3d.medir_distancia(A, B)]
        avisos.append(
            "Seção meridiana: triângulo de lados "
            f"{lados[0]:.2f}, {lados[1]:.2f} e {lados[2]:.2f}"
        )
    u3d.mostrar(fig)

    if desvio_do_vertice == 0:
        geratriz = math.sqrt(altura**2 + raio**2)  # Pitágoras no triângulo VOA
        # Com sliders, g = 2r quase nunca é exato: aceitamos uma diferença de até 2%.
        if abs(geratriz - 2 * raio) <= 0.02 * geratriz:
            print("Classificação: reto e equilátero (g ≈ 2r)")
        else:
            print("Classificação: reto")
        print(f"Geratriz: g = √({altura}² + {raio}²) = {geratriz:.2f}")
    else:
        print("Classificação: oblíquo")
        # Medimos as geratrizes até 360 pontos da circunferência da base.
        borda = u3d.poligono_regular(360, raio=raio)
        comprimentos = u3d.medir_distancia(borda, vertice)
        print(
            f"Geratrizes: da menor, {min(comprimentos):.2f}, "
            f"à maior, {max(comprimentos):.2f}"
        )
    for aviso in avisos:
        print(aviso)


interact(
    explorar_cone,
    raio=widgets.FloatSlider(
        min=0.5, max=3, step=0.25, value=2, description="raio",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=6, step=0.25, value=4, description="altura",
        continuous_update=False,
    ),
    desvio_do_vertice=widgets.FloatSlider(
        min=0, max=2, step=0.25, value=0, description="desvio de V",
        continuous_update=False,
    ),
    giro_da_secao=widgets.IntSlider(
        min=0, max=180, step=15, value=0, description="giro da seção (°)",
        continuous_update=False,
    ),
    secao=widgets.Dropdown(
        options=["Nenhuma", "Transversal", "Meridiana"],
        value="Nenhuma",
        description="seção",
    ),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='raio', max=3.0, min=0.5, st…

**Pare e pense**

1. Um cone reto tem raio 3 e altura 4. Quanto mede a geratriz?
2. Um cone equilátero tem raio 2. Quanto medem a geratriz e a altura?

<details>
<summary>Ver respostas</summary>

1. $g = \sqrt{16 + 9} = 5$.
2. $g = 2r = 4$ e $h = r\sqrt{3} = 2\sqrt{3} \approx 3{,}46$.
</details>

## 3. Planificação e áreas do cone reto

Corte a superfície lateral de um cone reto ao longo de uma geratriz e abra: ela vira um **setor circular** (o "chapéu" do gancho, desmontado).
- O **raio** do setor é a geratriz $g$.
- O **arco** do setor tem o comprimento da circunferência da base: $2\pi r$.

### O ângulo do setor

Numa circunferência de raio $g$, o arco é proporcional ao ângulo central. A circunferência inteira ($360°$) tem comprimento $2\pi g$, e o nosso arco tem $2\pi r$:
$$\frac{\theta}{360°} = \frac{2\pi r}{2\pi g} = \frac{r}{g} \qquad\Longrightarrow\qquad \boxed{\theta = 360° \cdot \frac{r}{g}} \quad \left(\text{em radianos: } \theta = \frac{2\pi r}{g}\right)$$

### As áreas

A área do setor também é proporcional ao ângulo. O disco inteiro de raio $g$ tem área $\pi g^2$:
$$S_\ell = \pi g^2 \cdot \frac{r}{g} \qquad\Longrightarrow\qquad \boxed{S_\ell = \pi r g}$$
$$S_b = \pi r^2 \qquad\qquad S_t = \pi r g + \pi r^2 = \pi r(g + r)$$

**Cone equilátero** ($g = 2r$): o setor é um **semicírculo** ($\theta = 180°$), $S_\ell = 2\pi r^2$ e $S_t = 3\pi r^2$.

> ⚠️ **Erro comum.** Usar a altura $h$ na fórmula da área lateral. A fórmula é $\pi r g$, com a **geratriz**: é ela que vira o raio do setor na planificação.

### A ponte com a pirâmide

Inscreva no cone uma pirâmide regular de $n$ lados. Sua área lateral é $p \cdot g_{\text{pir}}$ (notebook 10). Quando $n$ cresce, o semiperímetro $p$ se aproxima de $\pi r$ e o apótema da pirâmide se aproxima da geratriz $g$: a área lateral se aproxima de $\pi r g$.

**Aplicações:** o papel do chapéu ou da casquinha de sorvete, a cobertura de um telhado cônico (como o de uma torre de castelo), um funil de chapa de metal.

🕹️ Monte o cone pelas medidas e acompanhe a planificação.

In [5]:
def planificar_cone(raio: float, altura: float, enrolar: float) -> None:
    """Mostra a planificação do cone reto escolhido enrolando até formar o cone."""
    geratriz = math.sqrt(altura**2 + raio**2)
    angulo_do_setor = 360 * raio / geratriz
    area_lateral = math.pi * raio * geratriz
    area_total = math.pi * raio * (geratriz + raio)

    fig = u3d.criar_figura(titulo="A planificação do cone reto")
    fig = u3d.desenhar_enrolamento_do_cone(fig, geratriz, angulo_do_setor, enrolar)
    if enrolar == 0:
        # Visto de cima, o setor aparece em verdadeira grandeza.
        fig = u3d.definir_camera(fig, olho=(0, -0.4, 2.2))
    else:
        fig = u3d.definir_camera(fig, olho=(1.5, -1.5, 0.7))
    u3d.mostrar(fig)

    print(f"Geratriz: g = √({altura}² + {raio}²) = {geratriz:.2f}")
    print(f"Ângulo do setor: θ = 360° · {raio} / {geratriz:.2f} = {angulo_do_setor:.1f}°")
    print(f"Área lateral: πrg = {area_lateral:.2f}")
    print(f"Área total: πr(g + r) = {area_total:.2f}")


interact(
    planificar_cone,
    raio=widgets.FloatSlider(
        min=0.5, max=4, step=0.25, value=2, description="raio",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=6, step=0.25, value=4, description="altura",
        continuous_update=False,
    ),
    enrolar=widgets.FloatSlider(
        min=0, max=1, step=0.1, value=0, description="enrolar",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='raio', max=4.0, min=0.5, st…

Agora a ponte com a pirâmide, em números. Para o cone de raio 3 e altura 4, a tabela calcula a área lateral da pirâmide regular inscrita de $n$ lados:
- lado da base: $\ell = 2r\sin\frac{\pi}{n}$ (lembre que $\pi$ rad $= 180°$);
- apótema da base: $m = r\cos\frac{\pi}{n}$;
- apótema da pirâmide: $g_{\text{pir}} = \sqrt{h^2 + m^2}$ (notebook 10);
- área lateral: $\frac{n\ell}{2} \cdot g_{\text{pir}}$ (semiperímetro × apótema).

In [6]:
raio, altura = 3, 4
geratriz = math.sqrt(altura**2 + raio**2)

# A figura: o cone e uma pirâmide regular de 12 lados inscrita nele.
fig = u3d.criar_figura(titulo="Uma pirâmide de 12 lados inscrita no cone")
fig = u3d.desenhar_cone(fig, (0, 0, altura), (0, 0, 0), raio, opacidade=0.15)
inscrita = u3d.piramide(u3d.poligono_regular(12, raio=raio), (0, 0, altura))
fig = u3d.desenhar_poliedro(fig, *inscrita, opacidade=0.6)
u3d.mostrar(fig)

print("     n | área lateral da pirâmide inscrita")
for n in [3, 6, 12, 24, 48, 96, 1000]:
    lado = 2 * raio * math.sin(math.pi / n)
    apotema_da_base = raio * math.cos(math.pi / n)
    apotema_da_piramide = math.sqrt(altura**2 + apotema_da_base**2)
    area_lateral = n * lado / 2 * apotema_da_piramide
    print(f"{n:>6} | {area_lateral:.4f}")
print(f"  cone | {math.pi * raio * geratriz:.4f}   (πrg = 15π)")

     n | área lateral da pirâmide inscrita
     3 | 33.2970
     6 | 42.9273
    12 | 46.0223
    24 | 46.8451
    48 | 47.0540
    96 | 47.1064
  1000 | 47.1237
  cone | 47.1239   (πrg = 15π)


**Pare e pense**

1. Para o cone reto de raio 3 e altura 4, calcule o ângulo do setor, a área lateral e a área total.
2. Por que a planificação do cone equilátero é um semicírculo?

<details>
<summary>Ver respostas</summary>

1. $g = 5$; $\theta = 360° \cdot \frac{3}{5} = 216°$; $S_\ell = 15\pi \approx 47{,}12$; $S_t = 24\pi \approx 75{,}40$.
2. No cone equilátero, $g = 2r$, então $\theta = 360° \cdot \frac{r}{2r} = 180°$.
</details>

## 4. Volume do cone

**Teorema**
> O volume de um cone (reto ou oblíquo) é um terço do produto da área da base pela altura:
> $$\boxed{\text{volume} = \frac{1}{3}\pi r^2 h}$$

**Demonstração (com Cavalieri).** Coloque o cone e uma pirâmide com base de mesma área ($\pi r^2$) e mesma altura $h$ apoiados num mesmo plano. A uma distância $d$ do vértice, a seção do cone é um círculo de raio $r\frac{d}{h}$, com área $\pi r^2 \left(\frac{d}{h}\right)^2$; a seção da pirâmide tem área $S_b \left(\frac{d}{h}\right)^2 = \pi r^2 \left(\frac{d}{h}\right)^2$ (notebook 10). As áreas são iguais em todas as alturas. Pelo Princípio de Cavalieri, os volumes são iguais, e o volume da pirâmide é $\frac{1}{3}\pi r^2 h$. ∎

**Cone e cilindro.** Um cone tem **um terço** do volume do cilindro de mesma base e mesma altura: três cones enchem o cilindro, como três pirâmides enchiam o prisma.

**Cone equilátero** ($h = r\sqrt{3}$): volume $= \dfrac{\pi r^3\sqrt{3}}{3}$.

**Aplicações:** a casquinha de sorvete, a ampulheta, um funil, um **monte de areia ou de brita** (que naturalmente toma a forma de um cone), silos com fundo cônico.

🕹️ Compare as seções do cone e de uma pirâmide de mesma base e altura.

In [7]:
def comparar_cone_e_piramide(
    raio: float, altura: float, desvio_do_vertice: float, altura_do_corte: float
) -> None:
    """Corta um cone e uma pirâmide de mesma área da base e mesma altura pelo mesmo
    plano."""
    area_secao_cone = math.pi * (raio * (1 - altura_do_corte))**2
    lado_secao_piramide = raio * math.sqrt(math.pi) * (1 - altura_do_corte)
    area_secao_piramide = lado_secao_piramide**2
    volume_cone = math.pi * raio**2 * altura / 3
    volume_cilindro = math.pi * raio**2 * altura

    # A pirâmide reta de base quadrada fica à esquerda do cone, sem encostar nele.
    # Lado r·√π: a área da base é (r·√π)² = πr², a mesma do cone.
    lado_da_base = raio * math.sqrt(math.pi)
    raio_do_quadrado = lado_da_base / math.sqrt(2)
    x_da_piramide = -(raio + raio_do_quadrado + 1)
    base_da_piramide = u3d.poligono_regular(
        4, lado=lado_da_base, centro=(x_da_piramide, 0, 0)
    )
    piramide = u3d.piramide(base_da_piramide, (x_da_piramide, 0, altura))

    # As seções: o centro do círculo fica sobre o eixo VO, na altura do corte.
    z_do_corte = altura_do_corte * altura
    x_do_circulo = altura_do_corte * desvio_do_vertice
    circulo = u3d.poligono_regular(
        60, raio=raio * (1 - altura_do_corte), centro=(x_do_circulo, 0, z_do_corte)
    )
    quadrado = u3d.poligono_regular(
        4, lado=lado_secao_piramide, centro=(x_da_piramide, 0, z_do_corte)
    )

    fig = u3d.criar_figura(titulo="Mesma altura, mesma área da seção: mesmo volume")
    fig = u3d.desenhar_cone(
        fig, (desvio_do_vertice, 0, altura), (0, 0, 0), raio, opacidade=0.3
    )
    fig = u3d.desenhar_poliedro(fig, *piramide, opacidade=0.25)
    fig = u3d.desenhar_poligono(fig, circulo, cor=u3d.CORES["destaque"], opacidade=0.7)
    fig = u3d.desenhar_poligono(fig, quadrado, cor=u3d.CORES["destaque"], opacidade=0.7)
    # O plano de corte precisa ser largo o bastante para alcançar os dois sólidos.
    esquerda = x_da_piramide - raio_do_quadrado
    direita = max(raio, desvio_do_vertice)
    fig = u3d.desenhar_plano(
        fig, ((esquerda + direita) / 2, 0, z_do_corte), (0, 0, 1), opacidade=0.1,
        tamanho=direita - esquerda + 2,
    )
    u3d.mostrar(fig)

    print(f"Área da seção do cone: {area_secao_cone:.4f}")
    print(f"Área da seção da pirâmide: {area_secao_piramide:.4f}")
    print(f"Volume do cone: πr²h/3 = {volume_cone:.4f}")
    print(f"Volume do cilindro de mesma base e altura: πr²h = {volume_cilindro:.4f}")
    print(f"Razão entre os volumes: {volume_cone / volume_cilindro:.4f} (um terço)")

    # Conferência: uma pirâmide de 2000 lados inscrita no cone quase não difere dele.
    inscrita = u3d.piramide(
        u3d.poligono_regular(2000, raio=raio), (desvio_do_vertice, 0, altura)
    )
    print(f"Conferência (pirâmide inscrita de 2000 lados): {u3d.medir_volume(*inscrita):.4f}")


interact(
    comparar_cone_e_piramide,
    raio=widgets.FloatSlider(
        min=0.5, max=2.5, step=0.25, value=1.5, description="raio",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=4, description="altura",
        continuous_update=False,
    ),
    desvio_do_vertice=widgets.FloatSlider(
        min=0, max=2, step=0.25, value=1, description="desvio de V",
        continuous_update=False,
    ),
    altura_do_corte=widgets.FloatSlider(
        min=0, max=0.9, step=0.1, value=0.5, description="altura do corte",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=1.5, continuous_update=False, description='raio', max=2.5, min=0.5, st…

**Pare e pense**

1. Qual é o volume do cone reto de raio 3 e altura 4?
2. Um cone e um cilindro têm a mesma base e o mesmo volume. Qual é a relação entre as alturas?

<details>
<summary>Ver respostas</summary>

1. $\frac{1}{3}\pi \cdot 9 \cdot 4 = 12\pi \approx 37{,}70$.
2. A altura do cone é o triplo da altura do cilindro.
</details>

## Resumo

**Superfície cônica:** gerada por uma reta que passa por um ponto fixo (vértice) e se apoia numa curva (diretriz). Tem duas folhas. **Seções cônicas:** circunferência, elipse, parábola e hipérbole.

**Cone circular:** vértice $V$, base de raio $r$, eixo $VO$, geratrizes, altura $h$.
- **Reto** (de revolução): $g^2 = h^2 + r^2$; gerado por um triângulo retângulo girando em torno de um cateto.
- **Equilátero:** $g = 2r$, $h = r\sqrt{3}$ (seção meridiana equilátera).

**Seções:** transversal a distância $d$ do vértice = círculo de raio $r\frac{d}{h}$; meridiana (reto) = triângulo isósceles de base $2r$ e lados $g$.

**Planificação (cone reto):** setor de raio $g$ e ângulo $\theta = 360° \cdot \frac{r}{g}$ (ou $\frac{2\pi r}{g}$ rad).

**Áreas (cone reto):** $S_\ell = \pi r g$; $S_t = \pi r(g + r)$. Equilátero: $S_t = 3\pi r^2$.

**Volume (reto ou oblíquo):** $\frac{1}{3}\pi r^2 h$, um terço do cilindro de mesma base e altura.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Use `math.pi` nos cálculos.

O gabarito comentado está em [`0612b_cone_gabarito.ipynb`](0612b_cone_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) A seção meridiana de um cone reto é um triângulo isósceles.
# (b) No cone equilátero, a geratriz é o dobro do raio.
# (c) A planificação da superfície lateral de um cone reto é um setor circular.
# (d) O volume de um cone é metade do volume do cilindro de mesma base e mesma
#     altura.
# (e) Num cone reto, a geratriz é menor que a altura.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": True, "c": True, "d": False, "e": False}
dicas_ex1 = {
    "a": "a definição de seção meridiana (seção 2 - Cone circular)",
    "b": "a definição de cone equilátero (seção 2 - Cone circular)",
    "c": "a superfície lateral aberta (seção 3 - Planificação e áreas do cone reto)",
    "d": "a comparação entre cone e cilindro (seção 4 - Volume do cone)",
    "e": "a geratriz é a hipotenusa do triângulo VOA (seção 2 - Cone circular)",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex1[item]}."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: um cone reto tem raio 3 e altura 4. Calcule, em Python:
# (a) a geratriz;
# (b) o ângulo do setor da planificação, em graus;
# (c) a área lateral;
# (d) a área total;
# (e) o volume.
geratriz_ex2 = ...
angulo_ex2 = ...
area_lateral_ex2 = ...
area_total_ex2 = ...
volume_ex2 = ...

print(geratriz_ex2, angulo_ex2, area_lateral_ex2, area_total_ex2, volume_ex2)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", geratriz_ex2, 5, "g² = h² + r²", "seção 2 - Cone circular"),
    ("b", angulo_ex2, 216, "θ = 360° · r/g",
     "seção 3 - Planificação e áreas do cone reto"),
    ("c", area_lateral_ex2, 15 * math.pi, "S_ℓ = πrg",
     "seção 3 - Planificação e áreas do cone reto"),
    ("d", area_total_ex2, 24 * math.pi, "S_t = πr(g + r)",
     "seção 3 - Planificação e áreas do cone reto"),
    ("e", volume_ex2, 12 * math.pi, "volume = πr²h/3", "seção 4 - Volume do cone"),
]
for item, resposta, certa, dica, secao in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), {dica} ({secao})."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: um cone equilátero tem raio 2. Calcule, em Python:
# (a) a geratriz;
# (b) a altura;
# (c) o ângulo do setor da planificação, em graus;
# (d) a área total;
# (e) o volume.
geratriz_ex3 = ...
altura_ex3 = ...
angulo_ex3 = ...
area_total_ex3 = ...
volume_ex3 = ...

print(geratriz_ex3, altura_ex3, angulo_ex3, area_total_ex3, volume_ex3)

In [ ]:
# Verificação
respostas_ex3 = [
    ("a", geratriz_ex3, 4, "seção 2 - Cone circular"),
    ("b", altura_ex3, 2 * math.sqrt(3), "seção 2 - Cone circular"),
    ("c", angulo_ex3, 180, "seção 3 - Planificação e áreas do cone reto"),
    ("d", area_total_ex3, 12 * math.pi, "seção 3 - Planificação e áreas do cone reto"),
    ("e", volume_ex3, 8 * math.sqrt(3) * math.pi / 3, "seção 4 - Volume do cone"),
]
for item, resposta, certa, secao in respostas_ex3:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), lembre que no cone equilátero "
        f"g = 2r e h = r√3 ({secao})."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: o chapéu. Um chapéu de festa é feito com um setor
# circular de papel de raio 30 cm e ângulo central de 120°. Calcule:
# (a) o raio da base do chapéu;
# (b) a sua altura.
raio_ex4 = ...
altura_ex4 = ...

print(raio_ex4, altura_ex4)

In [ ]:
# Verificação
assert raio_ex4 is not ... and abs(raio_ex4 - 10) <= 0.01, (
    "Tente de novo, revise: no item (a), o arco do setor vira a circunferência da "
    "base: 2πr = 2π · 30 · 120/360 (seção 3 - Planificação e áreas do cone reto)."
)
assert altura_ex4 is not ... and abs(altura_ex4 - math.sqrt(800)) <= 0.01, (
    "Tente de novo, revise: no item (b), o raio do papel é a geratriz; use "
    "g² = h² + r² (seção 2 - Cone circular)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: a casquinha. Uma casquinha de sorvete tem a forma de um
# cone reto de raio 3 cm e altura 10 cm. Quantos cm³ de sorvete cabem dentro dela,
# nivelado com a borda?
volume_ex5 = ...

print(volume_ex5)

In [ ]:
# Verificação
assert volume_ex5 is not ... and abs(volume_ex5 - 30 * math.pi) <= 0.01, (
    "Tente de novo, revise: volume = πr²h/3 (seção 4 - Volume do cone)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: o monte de areia. Um monte de areia tem a forma de um
# cone com 4 m de diâmetro na base e 1,5 m de altura.
# (a) Qual é o volume de areia, em m³?
# (b) Quantas viagens um caminhão que leva 5 m³ precisa fazer para transportar todo
#     o monte?
volume_ex6 = ...
viagens_ex6 = ...

print(volume_ex6, viagens_ex6)

In [ ]:
# Verificação
assert volume_ex6 is not ... and abs(volume_ex6 - 2 * math.pi) <= 0.01, (
    "Tente de novo, revise: no item (a), o raio é 2 m e o volume é πr²h/3 "
    "(seção 4 - Volume do cone)."
)
assert viagens_ex6 == 2, (
    "Tente de novo, revise: no item (b), divida o volume por 5 e arredonde para cima "
    "com math.ceil (seção 4 - Volume do cone)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: classificando as cônicas. Complete a função
# classificar_conica, que recebe a semiabertura do cone e a inclinação do plano (em
# graus, em relação à horizontal) e devolve "circunferência", "elipse", "parábola" ou
# "hipérbole". Depois, use-a para classificar as seções de um cone de semiabertura
# 30° por planos inclinados 0°, 40°, 60° e 80°, guardando as respostas numa lista.
def classificar_conica(semiabertura: float, inclinacao_do_plano: float) -> str:
    """Classifica a seção de um cone duplo por um plano que não passa pelo vértice."""
    ...  # complete aqui: calcule o ângulo das geratrizes com a horizontal e compare-o
    # com a inclinação do plano


classificacoes_ex7 = ...  # complete aqui: a lista com as quatro classificações

print(classificacoes_ex7)

In [ ]:
# Verificação
assert classificacoes_ex7 == ["circunferência", "elipse", "parábola", "hipérbole"], (
    "Tente de novo, revise: compare a inclinação do plano com o ângulo das geratrizes "
    "com a horizontal, 90° − semiabertura (seção 1 - Superfícies cônicas)."
)
assert [classificar_conica(45, i) for i in (0, 30, 45, 70)] == [
    "circunferência", "elipse", "parábola", "hipérbole"
], (
    "Tente de novo, revise: a função precisa funcionar para qualquer semiabertura; "
    "use 90° − semiabertura, e não um número fixo (seção 1 - Superfícies cônicas)."
)
print("Certo!")

# As quatro seções, desenhadas como na seção 1.
for inclinacao in [0, 40, 60, 80]:
    i = math.radians(inclinacao)
    plano = ((0.5, 0, 1), (math.sin(i), 0, math.cos(i)))
    titulo = f"Plano inclinado {inclinacao}°: {classificar_conica(30, inclinacao)}"
    fig = u3d.criar_figura(titulo=titulo, altura=380)
    diretriz = u3d.poligono_regular(
        80, raio=3 * math.tan(math.radians(30)), centro=(0, 0, 3)
    )
    fig = u3d.desenhar_superficie_conica(
        fig, (0, 0, 0), diretriz + diretriz[:1], opacidade=0.25, prolongar=1
    )
    for trecho in u3d.secao_plana_do_cone_duplo(30, plano):
        fig = u3d.desenhar_curva_no_espaco(
            fig, trecho, cor=u3d.CORES["destaque"], espessura=6
        )
    u3d.mostrar(fig)

In [ ]:
# Exercício Desafio 8: o chapéu de maior volume. Você tem um disco de papel de raio
# 10 cm e vai recortar um setor para fazer um cone. Teste os ângulos do setor de 1° a
# 359°, de 0,1° em 0,1°, e encontre o ângulo que dá o cone de MAIOR volume. Guarde
# esse ângulo em angulo_ex8.
melhor_angulo = None
maior_volume = 0  # qualquer volume calculado será maior que zero

for k in range(10, 3591):
    angulo = k / 10
    ...  # complete aqui: calcule o raio, a altura e o volume do cone; se o volume for o
    # maior até agora, guarde-o em maior_volume e guarde o ângulo em melhor_angulo

angulo_ex8 = ...  # complete aqui

print(angulo_ex8)

In [ ]:
# Verificação
assert angulo_ex8 is not ... and abs(angulo_ex8 - 293.9) <= 1, (
    "Tente de novo, revise: para cada ângulo, calcule o raio r = 10 · ângulo/360, a "
    "altura h = √(100 − r²) e o volume πr²h/3 (seção 4 - Volume do cone)."
)
print("Certo!")
print(
    "Repare que o cone de maior volume não é nem o mais largo nem o mais alto. "
    "Encontrar máximos e mínimos de forma exata é um dos objetivos do Cálculo "
    "(módulo 11)."
)

## Conexões

**Isso depende de:**
- [`0606a_triedros_e_angulos_poliedricos.ipynb`](0606a_triedros_e_angulos_poliedricos.ipynb) (o defeito angular)
- [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb) (Princípio de Cavalieri)
- [`0610a_piramide.ipynb`](0610a_piramide.ipynb) (seções paralelas, área lateral e volume)
- [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb) (o primeiro corpo redondo; cone e cilindro de mesma base e altura)
- [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (comprimento da circunferência e de um arco) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (área do círculo e do setor circular)

**Isso será usado em:**
- [`0613a_esfera.ipynb`](0613a_esfera.ipynb) (a anticlépsidra, cilindro menos dois cones, e o volume da esfera)
- [`0614a_solidos_semelhantes_e_troncos.ipynb`](0614a_solidos_semelhantes_e_troncos.ipynb) (tronco de cone)
- [`0616a_inscricao_e_circunscricao_corpos_redondos.ipynb`](0616a_inscricao_e_circunscricao_corpos_redondos.ipynb) (esfera e cone, cone equilátero)
- 📌 Superfícies e sólidos de revolução (o cone gerado por um triângulo girando)
- 📌 Geometria Analítica (as cônicas: elipse, parábola e hipérbole)
- 📌 Limite, Integral e Derivada (máximos e mínimos)